In [1]:
import pandas as pd
import numpy as np
import joblib
from datetime import datetime

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import IsolationForest


# Load data
df = pd.read_csv("transactions_data.csv")
print("Dataset loaded:", df.shape)
display(df.head())


# Basic cleaning
df = df.drop_duplicates()

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].fillna("UNKNOWN")
    else:
        df[col] = df[col].fillna(df[col].median())

print("\nMissing values after cleaning:")
print(df.isnull().sum())


# Encode categorical columns
df_encoded = df.copy()
encoders = {}

for col in df_encoded.columns:
    if df_encoded[col].dtype == "object":
        encoder = LabelEncoder()
        df_encoded[col] = encoder.fit_transform(df_encoded[col])
        encoders[col] = encoder

print("\nEncoding done")


# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_encoded)

print("Scaling completed:", X_scaled.shape)


# Train model
model = IsolationForest(
    n_estimators=100,
    contamination=0.018,   # slightly tuned
    random_state=42
)

model.fit(X_scaled)
print("Model training finished")


# Detect anomalies
df_encoded["anomaly"] = model.predict(X_scaled)
df_encoded["anomaly_score"] = model.decision_function(X_scaled)

anomaly_count = (df_encoded["anomaly"] == -1).sum()
print(f"\nTotal anomalies detected: {anomaly_count}")

display(df_encoded[df_encoded["anomaly"] == -1].head())


# --- Explanation part ---
normal_data = df_encoded[df_encoded["anomaly"] == 1]

feature_columns = [col for col in df_encoded.columns 
                   if col not in ["anomaly", "anomaly_score", "reason"]]

feature_means = normal_data[feature_columns].mean()
feature_stds = normal_data[feature_columns].std()


def get_anomaly_reason(row, means, stds, threshold=2):
    reasons = []

    for col in means.index:
        if abs(row[col] - means[col]) > threshold * stds[col]:
            reasons.append(f"{col} deviates significantly")

    if not reasons:
        reasons.append("Unusual combination of features")

    return ", ".join(reasons)


df_encoded["reason"] = df_encoded.apply(
    lambda r: get_anomaly_reason(r, feature_means, feature_stds)
    if r["anomaly"] == -1 else "Normal",
    axis=1
)


print("\nSample explained anomalies:")
display(
    df_encoded[df_encoded["anomaly"] == -1][
        ["anomaly_score", "reason"]
    ].head()
)


# Save model
joblib.dump(model, "isolation_forest.pkl")
joblib.dump(scaler, "scaler.pkl")
print("\nModel and scaler saved")


# Save anomalies with timestamp
anomalies_df = df_encoded[df_encoded["anomaly"] == -1][
    ["anomaly_score", "reason"]
].copy()

anomalies_df["timestamp"] = datetime.now()

anomalies_df = anomalies_df[["timestamp", "anomaly_score", "reason"]]

anomalies_df.to_csv("anomalies.csv", index=False)

print("Anomalies saved to anomalies.csv")

Dataset loaded: (2512, 16)


,TransactionID,AccountID,TransactionAmount,TransactionDate,TransactionType,Location,DeviceID,IP Address,MerchantID,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,PreviousTransactionDate
0,TX000001,AC00128,14.09,11/04/23 16:29,Debit,San Diego,D000380,162.198.218.92,M015,ATM,70,Doctor,81,1,5112.21,04/11/24 8:08
1,TX000002,AC00455,376.24,27/06/23 16:44,Debit,Houston,D000051,13.149.61.4,M052,ATM,68,Doctor,141,1,13758.91,04/11/24 8:09
2,TX000003,AC00019,126.29,10/07/23 18:16,Debit,Mesa,D000235,215.97.143.157,M009,Online,19,Student,56,1,1122.35,04/11/24 8:07
3,TX000004,AC00070,184.50,05/05/23 16:32,Debit,Raleigh,D000187,200.13.225.150,M002,Online,26,Student,25,1,8569.06,04/11/24 8:09
4,TX000005,AC00411,13.45,16/10/23 17:51,Credit,Atlanta,D000308,65.164.3.100,M091,Online,26,Student,198,1,7429.40,04/11/24 8:06



Missing values after cleaning:
TransactionID              0
AccountID                  0
TransactionAmount          0
TransactionDate            0
TransactionType            0
Location                   0
DeviceID                   0
IP Address                 0
MerchantID                 0
Channel                    0
CustomerAge                0
CustomerOccupation         0
TransactionDuration        0
LoginAttempts              0
AccountBalance             0
PreviousTransactionDate    0
dtype: int64

Encoding done
Scaling completed: (2512, 16)
Model training finished

Total anomalies detected: 46


,TransactionID,AccountID,TransactionAmount,TransactionDate,TransactionType,Location,DeviceID,IP Address,MerchantID,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,PreviousTransactionDate,anomaly,anomaly_score
85,85,96,1340.19,2261,0,2,557,193,11,2,54,1,30,1,8654.28,0,-1,-0.030470
116,116,278,300.08,682,0,2,259,555,39,2,68,0,47,1,13546.75,6,-1,-0.003689
274,274,449,1176.28,1578,0,18,460,455,73,0,54,1,174,5,323.69,5,-1,-0.051778
340,340,105,1830.00,17,1,35,421,492,81,2,55,1,238,1,2235.70,5,-1,-0.010805
394,394,322,6.30,1093,1,8,522,330,16,1,80,2,283,5,7697.68,6,-1,-0.005745



Sample explained anomalies:


,anomaly_score,reason
85,-0.030470,TransactionAmount deviates significantly
116,-0.003689,AccountBalance deviates significantly
274,-0.051778,"TransactionAmount deviates significantly, Logi..."
340,-0.010805,TransactionAmount deviates significantly
394,-0.005745,"TransactionDuration deviates significantly, Lo..."



Model and scaler saved
Anomalies saved to anomalies.csv
